# Import libraries

In [1]:
import joblib
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from scipy.sparse import save_npz
from sentence_transformers import SentenceTransformer

from preprocessing_utils import (
    clean_missing,
    fix_identical_text_mislabels,
    dedupe_mirror_pairs,
    build_question_graph,
    get_connected_components,
    build_qid_to_component_map,
    split_components_into_train_val,
    assign_split_labels,
    add_shared_features,
    tokens_to_text,
    fit_tfidf_vectorizer,
    add_tfidf_cosine,
    fit_svd,
    add_svd_features,
    add_minimal_clean_columns,
    encode_questions,
    add_embedding_distance_features,
    determine_max_length,
    save_config,
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [18]:
RANDOM_STATE = 42
VAL_SIZE = 0.20
STOP_WORDS = set(stopwords.words("english"))
SVD_COMPONENTS = 100
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"
DEFAULT_BERT_CHECKPOINT = "distilbert-base-uncased"
DEFAULT_ROBERTA_CHECKPOINT = "FacebookAI/roberta-base"
PROCESS_DATA_PATH = "preprocess_data"

# Data loading

In [4]:
raw_df = pd.read_csv('quora_question_pairs_train.csv.zip', index_col=0)
raw_df.head()

,qid1,qid2,question1,question2,is_duplicate
id,,,,,
332278,459256,459257,The Iliad and the Odyssey in the Greek culture?,How do I prove that the pairs of three indepen...,0
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0
122738,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1


In [5]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 323432 entries, 332278 to 402019
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   qid1          323432 non-null  int64 
 1   qid2          323432 non-null  int64 
 2   question1     323431 non-null  object
 3   question2     323430 non-null  object
 4   is_duplicate  323432 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 14.8+ MB


# Data cleaning

## Removing rows with missing values

In [6]:
df = clean_missing(raw_df)

Deleted 3 rows


## Fixing labels for a pair of identical questions

In [7]:
df = fix_identical_text_mislabels(df)

question1_clean = df['question1'].str.strip().str.lower()
question2_clean = df['question2'].str.strip().str.lower()

is_identical = question1_clean == question2_clean

df[is_identical].sort_values(by='is_duplicate')

Label fixed for 5 examples


,qid1,qid2,question1,question2,is_duplicate
id,,,,,
359339,488974,480151,What is the meaning of this sentence below?,What is the meaning of this sentence below?,1
384501,162198,74460,What is the most beautiful moment in your life?,What is the most beautiful moment in your life?,1
257101,372357,372358,How do small business owners use CRM systems?,How do small business owners use CRM systems?,1
299649,422330,422331,Who is this?,Who is this?,1
233905,344263,125570,What is the best way to print a book of photos...,What is the best way to print a book of photos...,1
46601,83338,74190,Should I stop masturbating?,Should I stop masturbating?,1
70303,121195,121196,How much computer science does an 8 year old U...,How much computer science does an 8 year old U...,1
194310,294484,294485,How would you rephrase this sentence?,How would you rephrase this sentence?,1
115053,187684,41938,What are the chances I could be pregnant?,What are the chances I could be pregnant?,1


## Working with mirror cases of a question pair

In [8]:
df = dedupe_mirror_pairs(df)

Deleted 0 rows due to label conflict, 0 rows due to duplication


**Observation:** 
- After applying `clean_missing`, `fix_identical_text_mislabels`, and `dedupe_mirror_pairs`, the dataset reflects two data-quality issues identified in EDA: missing questions removed, identical-text mislabels corrected to `is_duplicate=1`, and mirror-pair conflicts/duplicates absent.


# Splitting data into training and test samples

In [9]:
graph = build_question_graph(df)

Number of nodes (unique questions): 449787
Number of edges (pairs of questions): 323429


In [10]:
components = get_connected_components(graph)

Number of components: 172694
Component size -- min: 2, median: 2.0, max: 3646
Component size distribution (percentiles):
count    172694.00
mean          2.60
std          10.43
min           2.00
50%           2.00
90%           3.00
99%          10.00
99.9%        36.00
max        3646.00
dtype: float64
Number of components with exactly 2 questions (one isolated pair): 136674 (79.1%)


In [11]:
qid_to_component = build_qid_to_component_map(components)

Mapped 449787 qid  to 172694 component


In [12]:
train_component_ids, val_component_ids = split_components_into_train_val(
        len(components), val_size=VAL_SIZE, random_state=RANDOM_STATE
    )

Number of components in train: 138155
Number of components in val: 34539


In [13]:
df = assign_split_labels(df, qid_to_component, val_component_ids)
df.head()

Number of rows in each split:
split
train    261296
val       62133
Name: count, dtype: int64
Actual share val (by rows): 19.2%

Class balance (is_duplicate proportion) in each split:
split
train    0.3730
val      0.3535
Name: duplicate_share, dtype: float64


,qid1,qid2,question1,question2,is_duplicate,split
id,,,,,,
332278,459256,459257,The Iliad and the Odyssey in the Greek culture?,How do I prove that the pairs of three indepen...,0,val
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train
122738,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1,train


In [14]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()

In [15]:
df.to_parquet(f"{PROCESS_DATA_PATH}/cleaned_with_split.parquet", index=False)
train_df.to_parquet(f"{PROCESS_DATA_PATH}/train_cleaned.parquet", index=False)
val_df.to_parquet(f"{PROCESS_DATA_PATH}/val_cleaned.parquet", index=False)

**Observation:** 
- The train/val split is done at the connected-component level, not per-row. This guarantees no question ever appears in both splits, which directly addresses the data-leakage risk flagged in the EDA conclusions. Non-duplicate questions repeat less often than duplicate ones, so a naive random split could let the model "memorize" a question rather than generalize.

# Feature engeneering

In [17]:
# with stopwords
train_df_sw = add_shared_features(train_df, remove_stopwords=False)
val_df_sw = add_shared_features(val_df, remove_stopwords=False)

print("Train samples")
train_df_sw.head(3)

Processed 261296 rows, remove_stopwords=False
Processed 62133 rows, remove_stopwords=False
Train samples


,qid1,qid2,question1,question2,is_duplicate,split,question1_tokens,question2_tokens,common_word_count,jaccard,common_ratio,len_diff_words,len_diff_chars
id,,,,,,,,,,,,,
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,"[what, be, practical, management, and, what, b...","[what, be, the, practical, aspect, of, strateg...",5,0.555556,0.833333,1,7
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,"[how, useful, be, makeuseof, answer]","[be, there, any, q, a, site, that, be, not, ya...",2,0.117647,0.400000,11,46
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,"[which, be, the, best, place, to, reside, in, ...","[which, ia, the, best, place, to, visit, in, i...",7,0.538462,0.777778,2,9


In [18]:
# without stopwords
train_df_nosw = add_shared_features(train_df, remove_stopwords=True)
val_df_nosw = add_shared_features(val_df, remove_stopwords=True)

print("Train samples")
train_df_nosw.head(3)

Processed 261296 rows, remove_stopwords=True
Processed 62133 rows, remove_stopwords=True
Train samples


,qid1,qid2,question1,question2,is_duplicate,split,question1_tokens,question2_tokens,common_word_count,jaccard,common_ratio,len_diff_words,len_diff_chars
id,,,,,,,,,,,,,
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,"[practical, management, strategic, management]","[practical, aspect, strategic, management]",3,0.750000,1.000000,0,7
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,"[useful, makeuseof, answer]","[q, site, yahoo, answer, hate, speech, allow]",1,0.111111,0.333333,4,46
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,"[best, place, reside, india]","[ia, best, place, visit, india]",3,0.500000,0.750000,1,9


In [19]:
train_df_sw.to_parquet(f"{PROCESS_DATA_PATH}/train_df_sw.parquet", index=False)
val_df_sw.to_parquet(f"{PROCESS_DATA_PATH}/val_df_sw.parquet", index=False)

train_df_nosw.to_parquet(f"{PROCESS_DATA_PATH}/train_df_nosw.parquet", index=False)
val_df_nosw.to_parquet(f"{PROCESS_DATA_PATH}/val_df_nosw.parquet", index=False)

# TF-IDF

## With stopwords

In [20]:
train_df_sw_tf_idf = tokens_to_text(train_df_sw)
val_df_sw_tf_idf = tokens_to_text(val_df_sw)

train_df_sw_tf_idf.head(3)

,qid1,qid2,question1,question2,is_duplicate,split,question1_tokens,question2_tokens,common_word_count,jaccard,common_ratio,len_diff_words,len_diff_chars,question1_clean,question2_clean
id,,,,,,,,,,,,,,,
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,"[what, be, practical, management, and, what, b...","[what, be, the, practical, aspect, of, strateg...",5,0.555556,0.833333,1,7,what be practical management and what be strat...,what be the practical aspect of strategic mana...
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,"[how, useful, be, makeuseof, answer]","[be, there, any, q, a, site, that, be, not, ya...",2,0.117647,0.400000,11,46,how useful be makeuseof answer,be there any q a site that be not yahoo answer...
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,"[which, be, the, best, place, to, reside, in, ...","[which, ia, the, best, place, to, visit, in, i...",7,0.538462,0.777778,2,9,which be the best place to reside in india and...,which ia the best place to visit in india


In [22]:
vectorizer_sw_tf_idf = fit_tfidf_vectorizer(train_df_sw_tf_idf)

dictionary size: 215129


In [23]:
train_df_sw_tf_idf, train_sw_tf_idf_q1_vec, train_sw_tf_idf_q2_vec = add_tfidf_cosine(train_df_sw_tf_idf, vectorizer_sw_tf_idf)
val_df_sw_tf_idf, val_sw_tf_idf_q1_vec, val_sw_tf_idf_q2_vec = add_tfidf_cosine(val_df_sw_tf_idf, vectorizer_sw_tf_idf)

train_df_sw_tf_idf.head()

,qid1,qid2,question1,question2,is_duplicate,split,question1_tokens,question2_tokens,common_word_count,jaccard,common_ratio,len_diff_words,len_diff_chars,question1_clean,question2_clean,tfidf_cosine
id,,,,,,,,,,,,,,,,
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,"[what, be, practical, management, and, what, b...","[what, be, the, practical, aspect, of, strateg...",5,0.555556,0.833333,1,7,what be practical management and what be strat...,what be the practical aspect of strategic mana...,0.500282
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,"[how, useful, be, makeuseof, answer]","[be, there, any, q, a, site, that, be, not, ya...",2,0.117647,0.400000,11,46,how useful be makeuseof answer,be there any q a site that be not yahoo answer...,0.060493
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,"[which, be, the, best, place, to, reside, in, ...","[which, ia, the, best, place, to, visit, in, i...",7,0.538462,0.777778,2,9,which be the best place to reside in india and...,which ia the best place to visit in india,0.318665
122738,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1,train,"[why, do, so, many, people, ask, question, on,...","[why, do, not, many, people, post, question, o...",9,0.281250,0.750000,19,91,why do so many people ask question on quora th...,why do not many people post question on quora ...,0.178593
250052,363829,363830,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0,train,"[i, get, a, seat, in, engineering, management,...","[i, be, a, petroleum, engineer, get, an, offer...",8,0.210526,0.363636,5,22,i get a seat in engineering management in kett...,i be a petroleum engineer get an offer from sc...,0.049684


In [24]:
train_df_sw_tf_idf.to_parquet(f"{PROCESS_DATA_PATH}/train_df_sw_tf_idf.parquet", index=False)
val_df_sw_tf_idf.to_parquet(f"{PROCESS_DATA_PATH}/val_df_sw_tf_idf.parquet", index=False)

In [25]:
save_npz(f"{PROCESS_DATA_PATH}/train_sw_tf_idf_q1_vec.npz", train_sw_tf_idf_q1_vec)
save_npz(f"{PROCESS_DATA_PATH}/train_sw_tf_idf_q2_vec.npz", train_sw_tf_idf_q2_vec)

save_npz(f"{PROCESS_DATA_PATH}/val_sw_tf_idf_q1_vec.npz", val_sw_tf_idf_q1_vec)
save_npz(f"{PROCESS_DATA_PATH}/val_sw_tf_idf_q2_vec.npz", val_sw_tf_idf_q2_vec)

In [26]:
joblib.dump(vectorizer_sw_tf_idf, f"{PROCESS_DATA_PATH}/vectorizer_sw_tf_idf.joblib")

['preprocess_data/vectorizer_sw_tf_idf.joblib']

### With SVD

In [27]:
svd_sw = fit_svd(train_sw_tf_idf_q1_vec, train_sw_tf_idf_q2_vec)

explained variance: 11.70%


In [28]:
train_df_sw_tf_idf_svd = add_svd_features(train_df_sw_tf_idf, train_sw_tf_idf_q1_vec, train_sw_tf_idf_q2_vec, svd_sw)
val_df_sw_tf_idf_svd = add_svd_features(val_df_sw_tf_idf, val_sw_tf_idf_q1_vec, val_sw_tf_idf_q2_vec, svd_sw)

train_df_sw_tf_idf_svd.head()

,qid1,qid2,question1,question2,is_duplicate,split,question1_tokens,question2_tokens,common_word_count,jaccard,...,q2_svd_90,q2_svd_91,q2_svd_92,q2_svd_93,q2_svd_94,q2_svd_95,q2_svd_96,q2_svd_97,q2_svd_98,q2_svd_99
0,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,"[what, be, practical, management, and, what, b...","[what, be, the, practical, aspect, of, strateg...",5,0.555556,...,0.002771,-0.000996,-0.000210,-0.011027,0.003078,-0.002213,-0.004684,0.000260,-0.008821,0.003365
1,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,"[how, useful, be, makeuseof, answer]","[be, there, any, q, a, site, that, be, not, ya...",2,0.117647,...,-0.000813,0.014216,-0.013116,-0.002396,0.022425,-0.016162,-0.015395,0.021079,0.012554,0.003694
2,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,"[which, be, the, best, place, to, reside, in, ...","[which, ia, the, best, place, to, visit, in, i...",7,0.538462,...,-0.006432,0.017445,0.014441,0.086373,-0.045393,0.026737,0.024533,-0.005883,-0.012869,-0.015971
3,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1,train,"[why, do, so, many, people, ask, question, on,...","[why, do, not, many, people, post, question, o...",9,0.281250,...,-0.000164,0.023096,-0.001361,0.018113,0.000566,0.005487,-0.004096,-0.046182,-0.002762,0.009809
4,363829,363830,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0,train,"[i, get, a, seat, in, engineering, management,...","[i, be, a, petroleum, engineer, get, an, offer...",8,0.210526,...,0.009745,0.010058,-0.029627,-0.004908,-0.009883,-0.024063,-0.016022,-0.007719,0.016859,0.013383


In [29]:
train_df_sw_tf_idf_svd.to_parquet(f"{PROCESS_DATA_PATH}/train_df_sw_tf_idf_svd.parquet", index=False)
val_df_sw_tf_idf_svd.to_parquet(f"{PROCESS_DATA_PATH}/val_df_sw_tf_idf_svd.parquet", index=False)

In [30]:
joblib.dump(svd_sw, f"{PROCESS_DATA_PATH}/svd_sw_transformer.joblib")

['preprocess_data/svd_sw_transformer.joblib']

## Without stopwords

In [31]:
train_df_nosw_tf_idf = tokens_to_text(train_df_nosw)
val_df_nosw_tf_idf = tokens_to_text(val_df_nosw)

train_df_nosw_tf_idf.head()

,qid1,qid2,question1,question2,is_duplicate,split,question1_tokens,question2_tokens,common_word_count,jaccard,common_ratio,len_diff_words,len_diff_chars,question1_clean,question2_clean
id,,,,,,,,,,,,,,,
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,"[practical, management, strategic, management]","[practical, aspect, strategic, management]",3,0.750000,1.000000,0,7,practical management strategic management,practical aspect strategic management
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,"[useful, makeuseof, answer]","[q, site, yahoo, answer, hate, speech, allow]",1,0.111111,0.333333,4,46,useful makeuseof answer,q site yahoo answer hate speech allow
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,"[best, place, reside, india]","[ia, best, place, visit, india]",3,0.500000,0.750000,1,9,best place reside india,ia best place visit india
122738,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1,train,"[many, people, ask, question, quora, easily, a...","[many, people, post, question, quora, check, g...",5,0.294118,0.625000,6,91,many people ask question quora easily answer n...,many people post question quora check google f...
250052,363829,363830,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0,train,"[get, seat, engineering, management, kettering...","[petroleum, engineer, get, offer, scotland, un...",4,0.200000,0.333333,0,22,get seat engineering management kettering univ...,petroleum engineer get offer scotland universi...


In [32]:
vectorizer_nosw_tf_idf = fit_tfidf_vectorizer(train_df_nosw_tf_idf)

dictionary size: 172154


In [33]:
train_df_nosw_tf_idf, train_nosw_tf_idf_q1_vec, train_nosw_tf_idf_q2_vec = add_tfidf_cosine(train_df_nosw_tf_idf, vectorizer_nosw_tf_idf)
val_df_nosw_tf_idf, val_nosw_tf_idf_q1_vec, val_nosw_tf_idf_q2_vec = add_tfidf_cosine(val_df_nosw_tf_idf, vectorizer_nosw_tf_idf)

train_df_nosw_tf_idf.head()

,qid1,qid2,question1,question2,is_duplicate,split,question1_tokens,question2_tokens,common_word_count,jaccard,common_ratio,len_diff_words,len_diff_chars,question1_clean,question2_clean,tfidf_cosine
id,,,,,,,,,,,,,,,,
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,"[practical, management, strategic, management]","[practical, aspect, strategic, management]",3,0.750000,1.000000,0,7,practical management strategic management,practical aspect strategic management,0.878978
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,"[useful, makeuseof, answer]","[q, site, yahoo, answer, hate, speech, allow]",1,0.111111,0.333333,4,46,useful makeuseof answer,q site yahoo answer hate speech allow,0.128030
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,"[best, place, reside, india]","[ia, best, place, visit, india]",3,0.500000,0.750000,1,9,best place reside india,ia best place visit india,0.225184
122738,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1,train,"[many, people, ask, question, quora, easily, a...","[many, people, post, question, quora, check, g...",5,0.294118,0.625000,6,91,many people ask question quora easily answer n...,many people post question quora check google f...,0.180450
250052,363829,363830,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0,train,"[get, seat, engineering, management, kettering...","[petroleum, engineer, get, offer, scotland, un...",4,0.200000,0.333333,0,22,get seat engineering management kettering univ...,petroleum engineer get offer scotland universi...,0.090578


In [34]:
train_df_nosw_tf_idf.to_parquet(f"{PROCESS_DATA_PATH}/train_df_nosw_tf_idf.parquet", index=False)
val_df_nosw_tf_idf.to_parquet(f"{PROCESS_DATA_PATH}/val_df_nosw_tf_idf.parquet", index=False)

In [35]:
joblib.dump(vectorizer_nosw_tf_idf, f"{PROCESS_DATA_PATH}/vectorizer_nosw_tf_idf.joblib")

['preprocess_data/vectorizer_nosw_tf_idf.joblib']

### With SVD

In [36]:
svd_nosw = fit_svd(train_nosw_tf_idf_q1_vec, train_nosw_tf_idf_q2_vec)

explained variance: 9.35%


In [37]:
train_df_nosw_tf_idf_svd = add_svd_features(train_df_nosw_tf_idf, train_nosw_tf_idf_q1_vec, train_nosw_tf_idf_q2_vec, svd_nosw)
val_df_nosw_tf_idf_svd = add_svd_features(val_df_nosw_tf_idf, val_nosw_tf_idf_q1_vec, val_nosw_tf_idf_q2_vec, svd_nosw)

train_df_nosw_tf_idf_svd.head()

,qid1,qid2,question1,question2,is_duplicate,split,question1_tokens,question2_tokens,common_word_count,jaccard,...,q2_svd_90,q2_svd_91,q2_svd_92,q2_svd_93,q2_svd_94,q2_svd_95,q2_svd_96,q2_svd_97,q2_svd_98,q2_svd_99
0,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,"[practical, management, strategic, management]","[practical, aspect, strategic, management]",3,0.750000,...,0.001743,-0.000522,-0.001499,-0.001979,0.002011,-0.002562,-0.001081,-0.000625,0.002287,0.003682
1,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,"[useful, makeuseof, answer]","[q, site, yahoo, answer, hate, speech, allow]",1,0.111111,...,0.008866,0.027980,0.005359,-0.007625,0.000451,-0.004390,-0.029462,0.008413,-0.012811,-0.004550
2,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,"[best, place, reside, india]","[ia, best, place, visit, india]",3,0.500000,...,0.001344,-0.006568,0.007986,0.014433,-0.022673,0.018463,0.014379,-0.005300,0.004824,-0.020186
3,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1,train,"[many, people, ask, question, quora, easily, a...","[many, people, post, question, quora, check, g...",5,0.294118,...,0.005284,0.026375,0.028975,-0.031191,-0.026575,0.021601,0.049008,-0.002874,0.040112,0.029171
4,363829,363830,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0,train,"[get, seat, engineering, management, kettering...","[petroleum, engineer, get, offer, scotland, un...",4,0.200000,...,-0.023749,0.002568,0.008160,-0.010970,-0.007091,0.014037,0.029514,0.010084,-0.011370,-0.000244


In [38]:
train_df_nosw_tf_idf_svd.to_parquet(f"{PROCESS_DATA_PATH}/train_df_nosw_tf_idf_svd.parquet", index=False)
val_df_nosw_tf_idf_svd.to_parquet(f"{PROCESS_DATA_PATH}/val_df_nosw_tf_idf_svd.parquet", index=False)

In [39]:
joblib.dump(svd_nosw, f"{PROCESS_DATA_PATH}/svd_nosw_transformer.joblib")

['preprocess_data/svd_nosw_transformer.joblib']

**Observation:** 
- SVD compresses the TF-IDF vectors from the full vocabulary size down to 100 dense components.The explained variance that confirms how much of the original signal is retained, is low. So it could be the tree-based models downstream may benefit more from the handcrafted overlap features than from the SVD components themselves.

# Sentence embeddings

## Model 1

In [44]:
train_df_se1 = add_minimal_clean_columns(train_df)
val_df_se1 = add_minimal_clean_columns(val_df)

In [45]:
model_1 = SentenceTransformer(EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [46]:
train_se_q1_emb1, train_se_q2_emb1 = encode_questions(train_df_se1, model_1)
val_se_q1_emb1, val_se_q2_emb1 = encode_questions(val_df_se1, model_1)

Batches:   0%|          | 0/1021 [00:00<?, ?it/s]

Batches:   0%|          | 0/1021 [00:00<?, ?it/s]

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

In [47]:
train_df_se1 = add_embedding_distance_features(train_df_se1, train_se_q1_emb1, train_se_q2_emb1)
val_df_se1 = add_embedding_distance_features(val_df_se1, val_se_q1_emb1, val_se_q2_emb1)

train_df_se1.head()

,qid1,qid2,question1,question2,is_duplicate,split,question1_clean,question2_clean,embedding_cosine,embedding_euclidean
id,,,,,,,,,,
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0.828576,0.585532
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0.061228,1.370235
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0.729282,0.735823
122738,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1,train,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,0.731709,0.732518
250052,363829,363830,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0,train,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0.501399,0.998600


In [48]:
embeddings_se1 = {
    "train_q1": train_se_q1_emb1, "train_q2": train_se_q2_emb1,
    "val_q1": val_se_q1_emb1, "val_q2": val_se_q2_emb1,
}

In [49]:
train_df_se1.to_parquet(f"{PROCESS_DATA_PATH}/train_df_se1.parquet", index=False)
val_df_se1.to_parquet(f"{PROCESS_DATA_PATH}/val_df_se1.parquet", index=False)

In [50]:
for name, arr in embeddings_se1.items():
    np.save(f"{PROCESS_DATA_PATH}/embedding_se1_{name}.npy", arr)
with open(f"{PROCESS_DATA_PATH}/embedding_model_name.txt", "w") as f:
    f.write(EMBEDDING_MODEL_NAME)

## Model 2

In [51]:
train_df_se2 = add_minimal_clean_columns(train_df)
val_df_se2 = add_minimal_clean_columns(val_df)

In [52]:
model_2 = SentenceTransformer(EMBEDDING_MODEL_NAME_2)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\ADMIN\ml_projects_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [53]:
train_se_q1_emb2, train_se_q2_emb2 = encode_questions(train_df_se2, model_2)
val_se_q1_emb2, val_se_q2_emb2 = encode_questions(val_df_se2, model_2)

Batches:   0%|          | 0/1021 [00:00<?, ?it/s]

Batches:   0%|          | 0/1021 [00:00<?, ?it/s]

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

Batches:   0%|          | 0/243 [00:00<?, ?it/s]

In [54]:
train_df_se2 = add_embedding_distance_features(train_df_se2, train_se_q1_emb2, train_se_q2_emb2)
val_df_se2 = add_embedding_distance_features(val_df_se2, val_se_q1_emb2, val_se_q2_emb2)

train_df_se2.head()

,qid1,qid2,question1,question2,is_duplicate,split,question1_clean,question2_clean,embedding_cosine,embedding_euclidean
id,,,,,,,,,,
196656,297402,297403,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0,train,What is practical management and what is strat...,What are the practical aspects of strategic ma...,0.856630,0.535481
113125,184949,184950,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0,train,How useful is MakeUseOf Answers?,Is there any Q&A site that is not Yahoo answer...,0.518023,0.981811
266232,101283,163744,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0,train,Which is the best place to reside in India and...,Which ia the best place to visit in India?,0.727708,0.737960
122738,17811,27517,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,1,train,Why do so many people ask questions on Quora t...,Why don't many people posting questions on Quo...,0.758127,0.695518
250052,363829,363830,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0,train,I got a seat in engineering management in Kett...,I am a petroleum engineer. Got an offer from S...,0.672337,0.809522


In [55]:
embeddings_se2 = {
    "train_q1": train_se_q1_emb2, "train_q2": train_se_q2_emb2,
    "val_q1": val_se_q1_emb2, "val_q2": val_se_q2_emb2,
}

In [56]:
train_df_se2.to_parquet(f"{PROCESS_DATA_PATH}/train_df_se2.parquet", index=False)
val_df_se2.to_parquet(f"{PROCESS_DATA_PATH}/val_df_se2.parquet", index=False)

In [57]:
for name, arr in embeddings_se2.items():
    np.save(f"{PROCESS_DATA_PATH}/embedding_se2_{name}.npy", arr)
with open(f"{PROCESS_DATA_PATH}/embedding_model_name_2.txt", "w") as f:
    f.write(EMBEDDING_MODEL_NAME_2)

**Observation:** 
- Two different sentence-transformer models are used to compare embedding quality downstream — `all-mpnet-base-v2` and `BAAI/bge-base-en-v1.5`.

# BERT prepearing

In [58]:
train_df_bert = add_minimal_clean_columns(train_df)
val_df_bert = add_minimal_clean_columns(val_df)

In [59]:
train_df_bert.to_parquet(f"{PROCESS_DATA_PATH}/train_df_bert.parquet", index=False)
val_df_bert.to_parquet(f"{PROCESS_DATA_PATH}/val_df_bert.parquet", index=False)

In [60]:
max_length = determine_max_length(train_df_bert, DEFAULT_BERT_CHECKPOINT)

99% percentile: 74 tokens
Recommended max_length: 80
count    261296.0
mean         30.4
std          12.5
min           6.0
50%          27.0
90%          46.0
95%          54.0
99%          74.0
max         248.0
dtype: float64


In [61]:
bert_config = {"model_checkpoint": DEFAULT_BERT_CHECKPOINT, "max_length": max_length}

save_config(bert_config, f"{PROCESS_DATA_PATH}/bert_config.json")

Saved: preprocess_data/bert_config.json


# RoBERTa prepearing

In [16]:
train_df_roberta = add_minimal_clean_columns(train_df)
val_df_roberta = add_minimal_clean_columns(val_df)

In [17]:
train_df_roberta.to_parquet(f"{PROCESS_DATA_PATH}/train_df_roberta.parquet", index=False)
val_df_roberta.to_parquet(f"{PROCESS_DATA_PATH}/val_df_roberta.parquet", index=False)

In [20]:
max_length_roberta = determine_max_length(train_df_roberta, DEFAULT_ROBERTA_CHECKPOINT)

99% percentile: 74 tokens
Recommended max_length: 80
count    261296.0
mean         31.0
std          12.2
min           7.0
50%          28.0
90%          47.0
95%          55.0
99%          74.0
max         264.0
dtype: float64


In [22]:
roberta_config = {"model_checkpoint": DEFAULT_ROBERTA_CHECKPOINT, "max_length": max_length_roberta}

save_config(roberta_config, f"{PROCESS_DATA_PATH}/roberta_config.json")

Saved: preprocess_data/roberta_config.json


**Observation:** 
- `max_length` is derived from the 99th percentile of tokenized pair length on train, not a fixed guess. This keeps sequences short enough for fast training without truncating the vast majority of question pairs.